In [2]:
# metrics_to_excel.py  ------------------------------------------------------
"""
计算 Classic vs. {任意方法} 的误差指标并写入 Excel。
可脚本执行，也可在 Jupyter 单元中运行整个文件内容。

默认评估：
    * density_MSE
    * l2_relative
    * fidelity
    * vorticity_MSE          （示例：对涡度评价）

如需扩展，请在 METRIC_FUNCS 字典里加自定义函数即可。
"""
import sys, pathlib, datetime
HERE = pathlib.Path(__file__).resolve().parent if "__file__" in globals() \
       else pathlib.Path().cwd()
sys.path.insert(0, str(HERE))                 # 同目录模块 import

import numpy as np, pandas as pd

from stateprep       import single_vortex
from spec_classic    import SpectralSolver    # 基准
from rk4_realspace   import RK4Solver         # 其它方法示例
from qspec_naive     import QuantumSpectralSolver
from qspec_pod       import QuantumPODSolver
from compute_utils   import compute_fluid_quantities

# ---------- 误差指标 ----------
def density_mse(ψa, ψb):
    ρa = np.abs(ψa[0])**2 + np.abs(ψa[1])**2
    ρb = np.abs(ψb[0])**2 + np.abs(ψb[1])**2
    return np.mean((ρa - ρb)**2)

def l2_relative(ψa, ψb):
    diff = np.linalg.norm(np.concatenate([ψa[0]-ψb[0], ψa[1]-ψb[1]]))
    ref  = np.linalg.norm(np.concatenate([ψb[0],   ψb[1]]))
    return diff / ref

def fidelity(ψa, ψb):
    flat_a = np.concatenate([ψa[0].ravel(), ψa[1].ravel()])
    flat_b = np.concatenate([ψb[0].ravel(), ψb[1].ravel()])
    return np.abs(np.vdot(flat_a, flat_b))**2

def vorticity_mse(ψa, ψb):
    *_, vor_a = compute_fluid_quantities(*ψa)
    *_, vor_b = compute_fluid_quantities(*ψb)
    return np.mean((vor_a - vor_b)**2)

METRIC_FUNCS = {
    "density_MSE"  : density_mse,
    "l2_relative"  : l2_relative,
    "fidelity"     : fidelity,
    "vorticity_MSE": vorticity_mse,
}

# ---------- 要比较的方法 ----------
METHODS = {
    "RK4"        : RK4Solver(),
    "Quantum"    : QuantumSpectralSolver(),
    "QuantumPOD" : QuantumPODSolver(),       # 若未实现会在 try/except 处理
}

# ---------- 物理/网格参数 ----------
T_GRID   = np.linspace(0, np.pi/2, 6)   # 6 time snapshots
SIGMA    = 3.0
CENTER   = (0., 0.)

# ================== 主流程 ==================
def main():
    # 1) 初态 & Classic 演化
    psi1_0, psi2_0 = single_vortex(SIGMA, CENTER)
    classic_solver = SpectralSolver()
    classic_data   = classic_solver.evolve(psi1_0, psi2_0, T_GRID)

    # Excel writer
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = HERE / f"metrics_{ts}.xlsx"
    writer   = pd.ExcelWriter(out_path, engine="openpyxl")

    # 2) 针对每个方法生成 DataFrame 并写 sheet
    for name, solver in METHODS.items():
        try:
            peer_data = solver.evolve(psi1_0, psi2_0, T_GRID)
        except NotImplementedError:
            print(f"[{name}] NotImplemented; skip.")
            continue

        rows = []
        for t in T_GRID:
            ψb = classic_data[t]
            ψp = peer_data[t]
            row = {"t": float(t)}
            for m_name, func in METRIC_FUNCS.items():
                row[m_name] = float(func(ψp, ψb))
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_excel(writer, sheet_name=name, index=False)
        print(f"✓ {name} -> sheet, shape {df.shape}")

    writer.close()
    print("▲ Metrics saved to", out_path)

# ------------------------------
if __name__ == "__main__":
    main()


✓ RK4 -> sheet, shape (6, 5)
✓ Quantum -> sheet, shape (6, 5)
✓ QuantumPOD -> sheet, shape (6, 5)
▲ Metrics saved to /home/adurey/QuantumComputing/fluidbench/metrics_20250802_114156.xlsx
